# RegQuest ML: 00 - System and Data Audit
### Ingestion, Integrity Verification, and Ground Truth Assessment
**Dataset:** 2025 Historical Request Logbooks (12 Monthly Registries, 9,265 Rows)

---
## 1. Objectives
1. Discover and inventory all historical logbook spreadsheets (`data/raw/`).
2. Audit row schemas, column headers, missing values, and corrupted text entries.
3. Evaluate ground-truth availability for credential processing turnaround times.


In [ ]:
import os, sys
from pathlib import Path
ML_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.regquest_ml.common.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.regquest_ml.data.loader import discover_raw_files, load_raw_csv_records

raw_files = discover_raw_files(str(RAW_DATA_DIR))
print(f'Discovered {len(raw_files)} monthly raw CSV files in {RAW_DATA_DIR}:')
for f in raw_files:
    print(f'  - {Path(f).name}')


In [ ]:
total_rows = 0
processing_time_non_null = 0
for fp in raw_files:
    rows = load_raw_csv_records(fp)
    total_rows += len(rows)
    for r in rows:
        pt = r.get('Processing Time ', r.get('Processing Time', '')).strip()
        if pt:
            processing_time_non_null += 1

print(f'Total historical records across 12 months: {total_rows:,}')
print(f'Records with explicit processing time: {processing_time_non_null} ({processing_time_non_null/total_rows*100:.2f}%)')
print('Insight: Only 2.7% of 2025 records contain a processing time entry.')
print('Conclusion: Historical logs do not yet record empirical physical completion timestamps.')
print('Solution: Use defensible administrative SLAs as baseline until production collects empirical timestamps.')
